In [16]:
import time
import json
import random
import concurrent.interpreters as interpreters

In [15]:

import sys
print(f"Version: {sys.version}")
print(f"Path: {sys.executable}")

try:
    import concurrent.interpreters as interpreters
    print("Success: concurrent.interpreters is available!")
except ImportError:
    print("Failure: concurrent.interpreters NOT found.")

Version: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Path: C:\Users\billo\AppData\Local\Python\pythoncore-3.14-64\python.exe
Success: concurrent.interpreters is available!


In [14]:
class DynamicMockNVML:
    def __init__(self, batch_size=32):
        self.step = 0
        self.batch_size = batch_size
        
        # Scaling Rule: 1 unit of Batch Size adds 3ms of compute
        # For BS=32, interval is ~100ms (10 samples @ 10ms each)
        self.samples_per_spike = max(5, int(self.batch_size * 0.3125))
        
    def get_power_usage(self):
        self.step += 1
        base_load = 200000 
        
        # The modulo now depends on the batch_size
        is_spike = (self.step % self.samples_per_spike == 0)
        spike_load = 500000 if is_spike else 0
        
        return base_load + spike_load + random.uniform(-5000, 5000)

In [19]:
def run_telemetry_sidecar(queue_id):
    batch_size = 64 # Try changing this to 32 or 128
    nvml_sim = DynamicMockNVML(batch_size=batch_size)
    nvml.nvmlInit()
    handle = nvml.nvmlDeviceGetHandleByIndex(0)
    queue = interpreters.get_queue(queue_id)
    
    while True:
        power_mw = nvml.nvmlDeviceGetPowerUsage(handle)
        power_w = power_mw / 1000.0
        
        # Send data to the main orchestrator
        queue.put({"wattage": power_w, "timestamp": time.time()})
        time.sleep(0.01) # 10ms sampling rate

# --- THE MAIN ORCHESTRATOR ---
if __name__ == "__main__":
    print("--- NIPO Infrastructure Prototype (Level 1) ---")
    
    # 1. Create the inter-interpreter Queue
    shared_queue = interpreters.create_queue()
    
    # 2. Spawn the parallel 'Sidecar' (Subinterpreter)
    sidecar = interpreters.create()
    print(f"Sidecar created. Monitoring Virtual GPU at 100Hz...")

    # 3. Process the data stream
    try:
        nvml_sim = DynamicMockNVML()
        
        while True:
            current_draw = nvml_sim.get_power_usage() / 1000.0

            if current_draw > 700:
                print(f"ALERT: Phase Spike Detected: {current_draw:.2f}W")
                print("ACTION: Flagging next cycle for Micro-Staggering...")
            
            time.sleep(0.01) 
    except KeyboardInterrupt:
        print("\nSimulation Terminated.")

--- NIPO Infrastructure Prototype (Level 1) ---
Sidecar created. Monitoring Virtual GPU at 100Hz...
ALERT: Phase Spike Detected: 704.68W
ACTION: Flagging next cycle for Micro-Staggering...
ALERT: Phase Spike Detected: 704.30W
ACTION: Flagging next cycle for Micro-Staggering...
ALERT: Phase Spike Detected: 700.04W
ACTION: Flagging next cycle for Micro-Staggering...
ALERT: Phase Spike Detected: 704.23W
ACTION: Flagging next cycle for Micro-Staggering...
ALERT: Phase Spike Detected: 702.46W
ACTION: Flagging next cycle for Micro-Staggering...
ALERT: Phase Spike Detected: 701.81W
ACTION: Flagging next cycle for Micro-Staggering...
ALERT: Phase Spike Detected: 702.84W
ACTION: Flagging next cycle for Micro-Staggering...
ALERT: Phase Spike Detected: 704.42W
ACTION: Flagging next cycle for Micro-Staggering...

Simulation Terminated.


Phase Locked Loop Predictor

In [21]:
batch_size = 64 # Try changing this to 32 or 128
nvml_sim = DynamicMockNVML(batch_size=batch_size)

class AdaptivePredictor:
    def __init__(self):
        self.history = []
        self.last_spike_time = time.time()
        self.current_interval = 0.1 # Default 100ms

    def record_spike(self):
        now = time.time()
        interval = now - self.last_spike_time
        self.last_spike_time = now
        # Moving average to smooth out noise
        self.current_interval = (self.current_interval * 0.8) + (interval * 0.2)
        return self.current_interval

# --- SIMULATION ---
print(f"Starting NIPO with Batch Size: {batch_size}")
print(f"Target Interval: {nvml_sim.samples_per_spike * 10}ms")

Starting NIPO with Batch Size: 64
Target Interval: 200ms
